# CPS Tutorial - Part 1 - Homomorphic Encryption

by Apostolos Fournaris (Industrial System Institute/R.C. ATHENA), Paolo Palmieri (UCC) and Francesco Regazzoni (UvA)

SECURED project (Horizon Europe grant no. 101095717) 

### The basics

The goal of this tutorial is to provide the students with some familiarity on how to work with Homomorphic Encryption (HE) libraries in order to create some meaningful application. 

We will focus on pi-HEaaN, a Python library for simulating HEAAN (Homomorphic Encryption for Arithmetic of Approximate Numbers), a fully homomorphic encryption scheme. Working with pi-HEaaN will enable you to gain experience of homomorphic encryption (HE). The library provides not only key generation, encryption and decryption, but homomorphic operations such as homomorphic addition and homomorphic multiplication.

pi-HEaaN performs encryption, decryption and homomorphic operations in block units. A single block consists of a maximum of 65,536 (2^16) slots.

Let's start by installing the library:

In [ ]:
#!pip install --upgrade pip
!pip install pi-heaan

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 34.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 24.2
[notice] To update, run: pip install --upgrade pip


And importing the required packages:

In [ ]:
import piheaan
import os

### Preliminaries
After importing the pi-hean library we can start setting the parameters and context for homomorphic encryption using the HEAAN scheme. You can find more info in https://heaan.it/

In [ ]:
params = piheaan.ParameterPreset.FGb
context = piheaan.make_context(params)
piheaan.make_bootstrappable(context)


### Key generation

For using homomorphic operations, it is necessary to generate secret and public keys. The public key is for encrypting a message, and result in a ciphertext (the message in encrypted form). We can perform homomorphic operations on the ciphertext. The secret key, on the other end, is for decrypting the ciphertext back to the plaintext (the unencrypted message).

In [ ]:
# Generate a folder with appropriate permissions to store the keys
key_file_path = "./key"
os.makedirs(key_file_path, mode=0o775, exist_ok=True)

# Generate the secret key 'sk'
sk = piheaan.SecretKey(context)

# Get the maximal logarithmic (base of 2) number of slots for the given context
log_num_slot = piheaan.get_log_full_slots(context)
num_slot = 1 << log_num_slot

# Save the secret key
sk.save(key_file_path+"/secretkey.bin")

# Load the secret key from file to make sure it works
sk = piheaan.SecretKey(context,key_file_path+"/secretkey.bin")

# Generate public (encryption/evaluation) keys from the secret key and save them
key_generator = piheaan.KeyGenerator(context, sk)
key_generator.gen_common_keys()
key_generator.save(key_file_path + "/")

### Get the generated key

In [ ]:
# Generate a KeyPack, which is an object to manage the public keys
keypack = piheaan.KeyPack(context, key_file_path+"/")

# Load encryption key from the file to the memory
keypack.load_enc_key()

# Load multiplication key from the file to the memory
keypack.load_mult_key()

### Generate HE utility objects

In [ ]:
# Generate homomorphic evaluation, encryption and decryption objects
eval = piheaan.HomEvaluator(context,keypack)
dec = piheaan.Decryptor(context)
enc = piheaan.Encryptor(context)

# Find available functions and arguments
print(dir(eval))
print(help(eval.add))
print(help(eval.mult))

['__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'add', 'bootstrap', 'bootstrap_extended', 'conjugate', 'i_mult', 'integer_mult', 'kill_imag', 'left_rotate', 'left_rotate_reduce', 'level_down', 'level_down_one', 'min_level_for_bootstrap', 'min_level_for_bootstrap_extended', 'mult', 'mult_without_rescale', 'negate', 'relinearize', 'rescale', 'right_rotate', 'right_rotate_reduce', 'rot_sum', 'square', 'sub', 'tensor']
Help on method add in module piheaan:

add(...) method of piheaan.HomEvaluator instance
    add(*args, **kwargs)
    Overloaded function.
    
    1. add(self: piheaan.HomEvaluator, op1: piheaan.Message, cnst: complex, res: piheaan.Message) -> None
    
    2. add(self: piheaan.HomEvaluator, op1: piheaan.Messa

### Generate messages and encrypt them

In [ ]:
# Create a plaintext full of 1s
one_ = [1]*num_slot
# Create an uninitialized message object
one_msg = piheaan.Message(log_num_slot)
# Set the object to the plaintext
for k in range(num_slot):
    one_msg[k] = one_[k]

# Create a ciphertext object to be used for the encrypted message
one_ctxt = piheaan.Ciphertext(context)
# Encrypt the plaintext into the ciphertext
enc.encrypt(one_msg, keypack, one_ctxt)

# Create a ciphertext full of 2s and repeat above steps
two_ = [2]*num_slot
two_msg = piheaan.Message(log_num_slot)
for k in range(num_slot):
    two_msg[k] = two_[k]

two_ctxt = piheaan.Ciphertext(context)
enc.encrypt(two_msg, keypack, two_ctxt)

### Perform a HE computation (addition)
Based on the above created ciphertexts we can now perform some HE operation on them. In the following example we perform an addition operation on the provided ciphertexts (i.e the encrypted versiob of the message full of ones and the encrypted message full of twos)

In [ ]:
# Create a ciphertext object for the result
result_add = piheaan.Ciphertext(context)

# Perform Homomorphic addition
eval.add(one_ctxt, two_ctxt, result_add)

#Create a message (plaintext) object for storing the decrypted result
add_msg = piheaan.Message(log_num_slot)

# Perform decryption 
dec.decrypt(result_add,sk,add_msg)

#Show the decrypted result
print(add_msg[:5])

[(3+0j), (3+0j), (3+0j), (3+0j), (3+0j)]


### Now, let's try multiplication

In [ ]:
# Now you should be expert enough to easily read the code below
result_mult = piheaan.Ciphertext(context)
eval.mult(two_ctxt, two_ctxt, result_mult)

mult_msg = piheaan.Message(log_num_slot)
dec.decrypt(result_mult,sk,mult_msg)

print(mult_msg[:5])

[(4+0j), (4+0j), (4+0j), (4+0j), (4+0j)]


# Finally, a more realistic scenario

Let's assume that we have a CPS client that needs to perform some computationally heavy operation but due to its low computation capabilities cannot handle such a task. The CPS will have to offload the computation to some Server device (eg. at the edge or cloud) and the server will then provide back to the CPS the result. However, the CPS client data to be processed is sensitive and privacy needs to be preserved. The CPS cannot release the data without privacy protection.  Thus the client uses Homomorphic Encryption to solve the privacy problem.

## Client side
#### (Initialization and preparation of the values for homomorphically encrypted computations )

In [ ]:
secret_val = 1

In [ ]:
import piheaan
import os

params = piheaan.ParameterPreset.FGb
context = piheaan.make_context(params)
piheaan.make_bootstrappable(context)

# Generate keys (see above)
key_file_path = "./key_ex"
os.makedirs(key_file_path, mode=0o775, exist_ok=True)
# Generate the secret key 'sk_ex'
sk_ex = piheaan.SecretKey(context)
# Get the maximal logarithmic (base of 2) number of slots for the given context
log_num_slot = piheaan.get_log_full_slots(context)
num_slot = 1 << log_num_slot
# Save the secret key
sk_ex.save(key_file_path+"/secretkey.bin")
# Load the secret key from file to make sure it works
sk_ex = piheaan.SecretKey(context,key_file_path+"/secretkey.bin")

# Generate public (encryption/evaluation) keys from the secret key and save them
key_generator_ex = piheaan.KeyGenerator(context, sk_ex)
key_generator_ex.gen_common_keys()
key_generator_ex.save(key_file_path + "/")

# Generate a KeyPack, which is an object to manage the public keys
keypack_ex = piheaan.KeyPack(context, key_file_path+"/")

# Load encryption key from the file to the memory
keypack_ex.load_enc_key()

# Load multiplication key from the file to the memory
keypack_ex.load_mult_key()

# Generate homomorphic evaluation, encryption and decryption objects
eval_ex = piheaan.HomEvaluator(context,keypack_ex)
dec_ex = piheaan.Decryptor(context)
enc_ex = piheaan.Encryptor(context)

# Create an uninitialized message object
val_msg = piheaan.Message(log_num_slot)

# Set message to secret value
val_msg[0] = secret_val

# Create a ciphertext object to be used for the encrypted message
val_ciphertxt = piheaan.Ciphertext(context)

# Encrypt the plaintext into the ciphertext
enc_ex.encrypt(val_msg, keypack_ex, val_ciphertxt)


## Server side 
#### (Perform Homomorphic Operation)
We can assume that the client has shared the public keys (as those are stored in the keypack) with the server. Also we assume that the client has transmitted the ciphertext to the server in order to be processed. The server will perform a complex operation that the CPS client due to low processing capabilities cannot handle.
In this toy example the server performs the operation f(x)=x<sup>3</sup>-23*x<sup>2</sup>+151*x-273. The function has the zero roots 3, 7 and 13.

The goal is for the students to provide the python code that performs this operation and stores the result in a heaan ciphertext object that is transmitted to the client.

In [ ]:
#provide code here


### Client side 
#### (Retrieval of the results)
The final action of the overall example is for the client to collect the encrypted result of the Server's computation and to retrieve from it the actual (plaintext) result. We can assume that the Server has sent the ciphertext to the client.

In [ ]:
# provide code here

# verify that the result is the expected one

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=e242c3b6-635b-4ae1-be8d-6a06c9f5d533' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>